# Inferential Statistics — Price Intelligence Platform
Tests statistical hypotheses about prices across sources and over time:
- One-way ANOVA (are mean prices equal across sources?)
- Post-hoc Tukey HSD (which pairs differ?)
- Kruskal-Wallis (non-parametric alternative when normality is rejected)
- Simple linear regression (price trend over time per source)

In [ ]:
# papermill injects values here
run_date = "2024-01-01"

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pingouin as pg
import plotly.express as px
import statsmodels.api as sm
from scipy import stats

print(f"Run date  : {run_date}")
print(f"pingouin  : {pg.__version__}")

In [ ]:
# ---------------------------------------------------------------------------
# Load data (same pattern as descriptive notebook)
# ---------------------------------------------------------------------------
def _load_from_jsonl(data_dir: str = "/opt/airflow/data") -> pd.DataFrame:
    frames = []
    for jsonl in Path(data_dir).rglob("*.jsonl"):
        frames.append(pd.read_json(jsonl, lines=True))
    if not frames:
        raise FileNotFoundError(f"No JSONL files found under {data_dir}")
    return pd.concat(frames, ignore_index=True)


try:
    sys.path.insert(0, "/opt/airflow")
    from bigtable.client import BigtableClient

    client = BigtableClient(
        project_id=os.environ.get("GCP_PROJECT_ID", "local"),
        instance_id=os.environ.get("BIGTABLE_INSTANCE_ID", "price-intelligence"),
    )
    rows = client._table.read_rows(limit=10000)
    records = [client._row_to_dict(r) for r in rows]
    df = pd.DataFrame(records)
    print(f"Loaded {len(df):,} rows from Bigtable")
except Exception as exc:
    print(f"Bigtable unavailable ({exc}) — loading from JSONL")
    df = _load_from_jsonl()
    print(f"Loaded {len(df):,} rows from JSONL files")

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["price"])
if "scraped_at" in df.columns:
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")
df.head()

In [ ]:
# ---------------------------------------------------------------------------
# 1. One-way ANOVA — H0: mean prices equal across all sources
# ---------------------------------------------------------------------------
groups = [g["price"].values for _, g in df.groupby("source")]
if len(groups) >= 2:
    f_stat, p_anova = stats.f_oneway(*groups)
    verdict = "REJECT H0 (prices differ)" if p_anova < 0.05 else "fail to reject H0"
    print(f"One-way ANOVA  F={f_stat:.4f}  p={p_anova:.4e}  → {verdict}")
else:
    print("Need ≥ 2 sources for ANOVA")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Post-hoc Tukey HSD (only if ANOVA rejects H0)
# ---------------------------------------------------------------------------
if len(groups) >= 2 and p_anova < 0.05:
    tukey = pg.pairwise_tukey(data=df, dv="price", between="source")
    print("\n=== Tukey HSD post-hoc ===")
    print(tukey[["A", "B", "mean(A)", "mean(B)", "diff", "p-tukey", "reject"]].to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# 3. Kruskal-Wallis (non-parametric, no normality assumption)
# ---------------------------------------------------------------------------
if len(groups) >= 2:
    h_stat, p_kw = stats.kruskal(*groups)
    verdict_kw = "REJECT H0" if p_kw < 0.05 else "fail to reject H0"
    print(f"\nKruskal-Wallis  H={h_stat:.4f}  p={p_kw:.4e}  → {verdict_kw}")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Price trend over time — OLS per source (requires scraped_at column)
# ---------------------------------------------------------------------------
if "scraped_at" in df.columns and df["scraped_at"].notna().any():
    df["ts_ordinal"] = df["scraped_at"].map(lambda x: x.toordinal() if pd.notna(x) else np.nan)
    print("\n=== OLS price trend (β₁ = price change per day) ===")
    for src, grp in df.groupby("source"):
        sub = grp.dropna(subset=["ts_ordinal", "price"])
        if len(sub) < 3:
            continue
        X = sm.add_constant(sub["ts_ordinal"])
        model = sm.OLS(sub["price"], X).fit()
        beta = model.params["ts_ordinal"]
        pval = model.pvalues["ts_ordinal"]
        r2 = model.rsquared
        print(f"  {src:30s}  β₁={beta:+.4f}/day  p={pval:.4e}  R²={r2:.3f}")
else:
    print("scraped_at column unavailable — skipping trend analysis")

In [ ]:
# ---------------------------------------------------------------------------
# 5. Violin plot — price spread per source
# ---------------------------------------------------------------------------
fig = px.violin(
    df,
    x="source",
    y="price",
    color="source",
    box=True,
    points="outliers",
    log_y=True,
    title=f"Price Violin Plot by Source ({run_date})",
)
fig.show()

print("\n✓ Inferential stats notebook complete")